# UCS420: Cognitive Computing
## Assignment 4 — A Cognitive FAQ System Using Pandas (Nova 2.0)

**Student Name:** Lakshay Kapur  
**Roll Number:** 1024170060  


### Q1: Build Your Personalized Knowledge Base
Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:
• Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", "general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g. if d%3 gives "account", write a question like “how do I update my registered mobile number”).
• # Example roll number ...23 -> digits 2, 3
• # digit 2 -> category[2 % 3] = general
• # digit 3 -> category[3 % 3] = billing

Output: Print your final 6-row DataFrame.

```python
fixed_entries = [
 {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
 "keywords": "fee cost price charge", "category": "billing"},
 {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
 "keywords": "password reset login", "category": "account"},
 {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
 "keywords": "hours timing open time", "category": "general"},
 {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
 "keywords": "pay payment upi fee", "category": "billing"},
]
```


In [1]:
import pandas as pd

# Roll number
roll_number = "1024170060"

# Last two digits
last_two_digits = roll_number[-2:]

# Category mapping
categories = ["billing", "account", "general"]

# Fixed FAQ entries
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

# Both digits (6 and 0) map to billing, so write two distinct billing FAQs.
personalized_faqs = [
    ("how can i check my payment status",
     "You can check your payment status from the Billing section.",
     "payment status billing"),
    ("where can i download my receipt",
     "Open Billing and download the receipt for the completed payment.",
     "receipt invoice download"),
]
personalized_entries = [
    {
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": categories[int(digit) % 3]
    }
    for digit, (question, answer, keywords) in zip(last_two_digits, personalized_faqs)
]

# Create final 6-row DataFrame
df = pd.DataFrame(fixed_entries + personalized_entries)

print("Final FAQ Knowledge Base (6 entries):")
display(df)


Final FAQ Knowledge Base (6 entries):


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my payment status,You can check your payment status from the Bil...,payment status billing,billing
5,where can i download my receipt,Open Billing and download the receipt for the ...,receipt invoice download,billing


### Q2: Generate and Score a Hypothesis
Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.


In [2]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        text_words = set(text.split())

        score = len(query_words.intersection(text_words))

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    results = sorted(
        results,
        key=lambda x: x["confidence"],
        reverse=True
    )

    return pd.DataFrame(results, columns=["question", "answer", "category", "confidence"])

test_query = "how can i pay annual fee online"
print(f"Scoring query: '{test_query}'")
results = score_query(test_query, df)

print("\nMatching FAQ entries ranked by confidence:")
display(results)


Scoring query: 'how can i pay annual fee online'

Matching FAQ entries ranked by confidence:


,question,answer,category,confidence
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,5
1,how can i check my payment status,You can check your payment status from the Bil...,billing,3
2,what is the annual fee,The annual fee is Rs 500.,billing,2
3,where can i download my receipt,Open Billing and download the receipt for the ...,billing,2
4,how to reset password,Go to Settings > Reset Password.,account,1


### Q3: Filter by Category
Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.


In [3]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

# Category of the first personalized entry
personalized_category = personalized_entries[0]["category"]

print("Selected personalized category:", personalized_category)
print("\nQuestions belonging to this category:")

category_results = same_category(personalized_category, df)
display(category_results[["question", "category"]])


Selected personalized category: billing

Questions belonging to this category:


,question,category
0,what is the annual fee,billing
3,how can i pay the fee,billing
4,how can i check my payment status,billing
5,where can i download my receipt,billing


### Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named `<your_roll_number>_faq_data.csv`.


In [4]:
# Display current entries
print("Current FAQ Knowledge Base:")
display(df[["question", "keywords", "category"]])

# Select entry 0 to update with a new keyword
entry_index = 0
new_keyword = input("Enter a new keyword for entry 0: ").strip().lower()
while len(new_keyword.split()) != 1 or new_keyword in df.loc[entry_index, "keywords"].split():
    new_keyword = input("Enter one keyword that is not already present: ").strip().lower()

print(f"\nAdding new keyword '{new_keyword}' to entry at index {entry_index}...")
df.loc[entry_index, "keywords"] = df.loc[entry_index, "keywords"] + " " + new_keyword

# Save updated DataFrame to CSV
filename = f"{roll_number}_faq_data.csv"
df.to_csv(filename, index=False)

print(f"\nUpdated entry {entry_index}:")
print(df.loc[entry_index].to_string())

print(f"\nUpdated DataFrame successfully saved as: {filename}")


Current FAQ Knowledge Base:


,question,keywords,category
0,what is the annual fee,fee cost price charge,billing
1,how to reset password,password reset login,account
2,what are your working hours,hours timing open time,general
3,how can i pay the fee,pay payment upi fee,billing
4,how can i check my payment status,payment status billing,billing
5,where can i download my receipt,receipt invoice download,billing


Enter a new keyword for entry 0: 
Enter one keyword that is not already present: fee
Enter one keyword that is not already present: charges

Adding new keyword 'charges' to entry at index 0...

Updated entry 0:
question           what is the annual fee
answer          The annual fee is Rs 500.
keywords    fee cost price charge charges
category                          billing

Updated DataFrame successfully saved as: 1024170060_faq_data.csv


### Q5: Group by Category
Using groupby, print how many FAQ entries you have per category.


In [5]:
category_counts = df.groupby("category").size().reset_index(name="count")

print("Number of FAQ entries per category:")
display(category_counts)


Number of FAQ entries per category:


,category,count
0,account,1
1,billing,4
2,general,1


### Q6: Handle Ties in Hypothesis Scoring
Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.


In [6]:
def score_query_with_ties(query, df):
    ranked = score_query(query, df)
    if ranked.empty:
        print(f"Query: {query!r}. No matching FAQ entry found.")
        return ranked

    highest_score = ranked["confidence"].iloc[0]
    best_matches = ranked.loc[ranked["confidence"] == highest_score]
    print(f"Query: {query!r} | highest confidence: {highest_score}")
    display(best_matches)
    return best_matches

print("========== TIE DEMONSTRATION ==========")
score_query_with_ties("fee", df)

print("\n========== NON-TIE DEMONSTRATION ==========")
score_query_with_ties("password", df)


========== TIE DEMONSTRATION ==========
Query: 'fee' | highest confidence: 1


,question,answer,category,confidence
0,what is the annual fee,The annual fee is Rs 500.,billing,1
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1



========== NON-TIE DEMONSTRATION ==========
Query: 'password' | highest confidence: 1


,question,answer,category,confidence
0,how to reset password,Go to Settings > Reset Password.,account,1
